# Семинар 1. Линейная регрессия

#### Шаг 1. Загрузка датасета и вывод на экран

In [ ]:
# Ваш код
import pandas as pd

# Загрузка датасета
df = pd.read_csv('delivery_dataset.csv')

# Просмотр первых строк и общей информации
display(df.head())
df.info()

#### Шаг2. Разделение выборки на тренировочную, валидационную и тестовую с выбором нужных столбцов:
* в тренировочной: 300 первых заказов;
* в валидационной: 100 следующих; 
* в тестовой: 100 последних.

In [ ]:
# Ваш код
# Выбираем числовые признаки для X и целевую переменную y
feature_cols = [
    'Расстояние до клиента (в км)',
    'Количество позиций в чеке (в шт.)',
    'Балл пробок на дорогах (в баллах)',
    'Погода (в градусах Цельсия)',
    'Этаж доставки (в этажах)'
]
target_col = 'Время доставки (в минутах)'

X = df[feature_cols]
y = df[target_col]

# Разделение по срезам: 300 train, 100 val, 100 test
X_train, y_train = X.iloc[:300], y.iloc[:300]
X_val, y_val = X.iloc[300:400], y.iloc[300:400]
X_test, y_test = X.iloc[400:], y.iloc[400:]

# Проверка размеров
print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Val:   X={X_val.shape}, y={y_val.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")

#### Вопрос 1. 

Для чего мы разбиваем данные на выборки? Для чего нам нужна тренировачная, для чего валидационная, для чего тестовая?

В тренировочном алгоритм подбирает веса. Валидационая нужна для того, чтобы контролировать процесс обучения. Мы смотрим, не переобучилась ли модель, или наоборот недоучилась. А тестовая нужна для контрольной проверки в самом конце, чтобы понять, как наша модель будет работь на уже реальной практике

#### Шаг 3. Установка sklearn чтобы обучать модель линейной регрессии через Ridge

In [3]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Шаг 4. Обучение модели линейной регрессии на тренировочной выборке с помощью Ridge и sparse_cg

In [ ]:
# Ваш код
from sklearn.linear_model import Ridge

# Инициализация модели без регуляризации с солвером sparse_cg
model = Ridge(alpha=0.0, solver='sparse_cg')

# Обучение на обучающей выборке
model.fit(X_train, y_train)

# Вывод полученных весов
print("Свободный член (w0 / intercept):", model.intercept_)
print("Коэффициенты (w1..w5):", model.coef_)

#### Шаг 5. Вывод на экран графиков ошибок (SSE, MSE, RMSE) в зависимости от итераций для train и val выборок.

In [ ]:
# Ваш код
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

iterations = list(range(1, 11))

train_sse, val_sse = [], []
train_mse, val_mse = [], []
train_rmse, val_rmse = [], []

for it in iterations:
    # Обучаем модель с ограничением на количество итераций
    m = Ridge(alpha=0.0, solver='sparse_cg', max_iter=it, tol=1e-12)
    m.fit(X_train, y_train)

    pred_train = m.predict(X_train)
    pred_val = m.predict(X_val)

    # Расчет SSE
    sse_tr = np.sum((y_train - pred_train) ** 2)
    sse_v = np.sum((y_val - pred_val) ** 2)

    # Расчет MSE
    mse_tr = mean_squared_error(y_train, pred_train)
    mse_v = mean_squared_error(y_val, pred_val)

    # Расчет RMSE
    rmse_tr = np.sqrt(mse_tr)
    rmse_v = np.sqrt(mse_v)

    train_sse.append(sse_tr)
    val_sse.append(sse_v)
    train_mse.append(mse_tr)
    val_mse.append(mse_v)
    train_rmse.append(rmse_tr)
    val_rmse.append(rmse_v)

# Построение графиков
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# График SSE
axes[0].plot(iterations, train_sse, marker='o', label='Train SSE')
axes[0].plot(iterations, val_sse, marker='s', label='Val SSE')
axes[0].set_title('SSE в зависимости от итераций')
axes[0].set_xlabel('Итерация')
axes[0].set_ylabel('SSE')
axes[0].grid(True)
axes[0].legend()

# График MSE
axes[1].plot(iterations, train_mse, marker='o', label='Train MSE')
axes[1].plot(iterations, val_mse, marker='s', label='Val MSE')
axes[1].set_title('MSE в зависимости от итераций')
axes[1].set_xlabel('Итерация')
axes[1].set_ylabel('MSE')
axes[1].grid(True)
axes[1].legend()

# График RMSE
axes[2].plot(iterations, train_rmse, marker='o', label='Train RMSE')
axes[2].plot(iterations, val_rmse, marker='s', label='Val RMSE')
axes[2].set_title('RMSE в зависимости от итераций')
axes[2].set_xlabel('Итерация')
axes[2].set_ylabel('RMSE')
axes[2].grid(True)
axes[2].legend()

plt.tight_layout()
plt.show()

#### Вопрос 2.

Какую информацию про нашу модель несут эти графики?

По графикам видно, за сколько итераций модель приходит к оптимальным весам. Также по ним мы понимаем, как воообще обучилась наша модель (обучилась, недообучилась, переобучилась). Также видно различие между разными видами ошибок. Например в SSE такой большой разброс между двумя кривыми, потому что разный объём выборки у train и у val (300 и 100). А MSE и RMSE уже усреднены, поэтому кривые уже находятся рядом друг к другу

#### Вопрос 3. 

Какая траектория линий train и val ошибок будет на графиках недообученной модели и почему?

Какая траектория линий train и val ошибок будет на графиках обученной модели и почему?

Какая траектория линий train и val ошибок будет на графиках переобученной модели и почему?

При недообучении обе линии становятся пологими (горизонтальными), и при этом высокое значение ошибки. Это может происхоить если модель простая, не хватает признаков или сложности
При обучении обе линии спадают и затем уже становятся пологими, значение ошибки близко к 0. Здесь модель нашла зависимости  и одинаково хорошо работает на обеих выборках
При переобучении train падает почти до нуля, а вот val наоборот растёт и находится на высоком знаенчии ошибки. Здесь модель выучила весь случайный шум обучающей выборки и не имеет способности обобщать новые данные

#### Шаг 6. Применение модели на тестовой выборке

In [ ]:
# Ваш код
# Предсказание на тестовой выборке
y_test_pred = model.predict(X_test)

# Вывод первых 5 предсказанных и реальных значений
test_results = pd.DataFrame({
    'Реальное время': y_test.values[:5],
    'Предсказанное время': y_test_pred[:5].round(1)
})
display(test_results)

#### Шаг 7. RMSE на тестовой выборке

In [ ]:
# Ваш код
from sklearn.metrics import mean_squared_error
import numpy as np

# Расчет RMSE на тестовой выборке
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"RMSE на тесте: {test_rmse:.4f} минут")

#### Вопрос 4.

Что получившийся RMSE может сказать о нашей модели?

RMSE показывает средний разброс ошибки. В данной задаче практический смысл такой: если сказали, что заказ привезут за 1 час, то в среднем реальное время будет откланятся на +-RMSE минут